In [ ]:
import sys, subprocess, importlib

# Packages needed for the project
REQUIRED = {
    "gudhi": "gudhi",
    "cv2": "opencv-python",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
}

missing = []
for module_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print("Missing packages detected:", missing)
    print("Installing now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Installation complete.")
else:
    print("All required libraries are already installed.")

# Printing versions of the libraries
import os
import glob # for file pattern recognisation
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import gudhi as gd
import cv2
from gudhi.representations import PersistenceImage
from tqdm import tqdm
import random

print("\nVersions:")
print("Python:", sys.version.split()[0])
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)
print("gudhi:", gd.__version__)

This notebook uses Google Drive to load the dataset.
Please ensure access to the shared TDAPRACTICUMDATASET folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/JITIN_KANUMALLI_TDAPRACTICUMDATASET"

In [ ]:
print("Yes folder exists:", os.path.exists(DATASET_DIR + "/yes")) #checking whether folder exists or not
print("No folder exists :", os.path.exists(DATASET_DIR + "/no"))

In [ ]:
YES_DIR = DATASET_DIR + "/yes"
NO_DIR  = DATASET_DIR + "/no"

valid_ext = (".jpg", ".jpeg", ".png", ".bmp")

yes_paths = sorted([p for p in glob.glob(YES_DIR + "/*") if p.lower().endswith(valid_ext)])
no_paths  = sorted([p for p in glob.glob(NO_DIR + "/*") if p.lower().endswith(valid_ext)])

print("Tumor images:", len(yes_paths))
print("No tumor images:", len(no_paths))

In [ ]:
X_paths = yes_paths + no_paths
y = np.array([1]*len(yes_paths) + [0]*len(no_paths))  # 1=tumor, 0=no tumor as lables

print("Total images:", len(X_paths))
print("Label vector shape:", y.shape)

In [ ]:
def show_samples(paths, labels, n=6):
    plt.figure(figsize=(12,3))
    for i in range(n):
        img = cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE)
        plt.subplot(1,n,i+1)
        plt.imshow(img, cmap='gray')
        plt.title("Tumor" if labels[i]==1 else "No tumor")
        plt.axis('off')
    plt.show()

show_samples(X_paths, y)

In [ ]:
def image_to_cubical_complex(image_gray):
    """
    Convert grayscale MRI image to cubical complex using vertex filtration.
    """
    # convert to float
    image_array = np.array(image_gray, dtype=np.float32)

    # normalize to [0,1] so that we can ensure that filtration values are comparable.
    image_array = (image_array - image_array.min()) / (image_array.max() - image_array.min() + 1e-8)

    # invert bright tumor areas appear early in filtration
    filtration_image = 1.0 - image_array

    # pad vertices so cubical complex can be built
    vertex_array = np.pad(image_array, pad_width=1, mode='edge')

    # build cubical complex
    cc = gd.CubicalComplex(vertices=vertex_array)

    return cc, image_array, filtration_image, vertex_array

In [ ]:
# pick one tumor image
sample_path = yes_paths[0]

# load grayscale image
image_gray = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)

# build cubical complex
cc, image_array, filtration_image, vertex_array = image_to_cubical_complex(image_gray)

print("Original image shape:", image_gray.shape)
print("Filtration image shape:", filtration_image.shape)
print("Vertex array shape:", vertex_array.shape)
print("Cubical complex dimension:", cc.dimension())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# original MRI
axes[0].imshow(image_array, cmap='gray')
axes[0].set_title("Original MRI")
axes[0].axis('off')

# filtration image
axes[1].imshow(filtration_image, cmap='gray')
axes[1].set_title("Filtration Image (1 - intensity)")
axes[1].axis('off')

Normalize image
Ensures filtration values are comparable.

Invert intensity
Bright tumor regions appear early in filtration.

Pad vertices
GUDHI requires vertex grid one unit larger than pixel grid.

In [ ]:
def compute_persistence(cc):
    cc.compute_persistence(homology_coeff_field=2)

    H0 = cc.persistence_intervals_in_dimension(0)
    H1 = cc.persistence_intervals_in_dimension(1)

    return H0, H1

GUDHI internally computes persistent cohomology, which returns:

H0 intervals → connected components

H1 intervals → holes/loops

In [ ]:
# persistence image transformer
persistence_image = PersistenceImage(
    bandwidth=0.1,
    resolution=[20,20],
    im_range=[0,1,0,1]
)

1.  bandwidth: Gaussian smoothing Each point in the diagram becomes a Gaussian
(blurred dot)
2.  resolution:	size of resulting vector  
3.im_range:	birth/persistence range



each diagram → 400-dimensional vector (20×20)                           [x_min, x_max, y_min, y_max]          [0,1,0,1] means:

birth values from 0 to 1
persistence values from 0 to 1

In [ ]:
def diagram_to_vector(diagram):
    if not len(diagram):
        return np.zeros(400)

    # GUDHI handles the [birth, death] -> [birth, persistence] conversion internally
    return persistence_image.fit_transform([np.array(diagram)])[0]

diagram = list of points like [(birth,death), (birth,death), ...] persistence = death - birth and then we transform
(birth, death) → (birth, persistence)

Persistence Diagram → Feature Vector

In [ ]:
def process_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    cc_tuple = image_to_cubical_complex(img)
    cc = cc_tuple[0] # Extract the CubicalComplex object

    H0, H1 = compute_persistence(cc)

    vec_H0 = diagram_to_vector(H0)
    vec_H1 = diagram_to_vector(H1)

    # combine features
    features = np.concatenate([vec_H0, vec_H1])

    return features

by diagram_to_vector(diagram) we get vector like [0.00, 0.02, 0.15, 0.00, 0.33, ..., 0.01] each for h0 and h1 and them we combine both of them. Here each vector acts as the feature so we get total of 400+400 =800 features. each value in a vector represents intensity of topological feature in the persistence image

In [ ]:
X = []
y = []

# Process tumor images
for path in tqdm(yes_paths, desc="Processing tumor images"):
    X.append(process_image(path))
    y.append(1)

# Process non-tumor images
for path in tqdm(no_paths, desc="Processing non-tumor images"):
    X.append(process_image(path))
    y.append(0)

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

In [ ]:
path = yes_paths[0]

# load image
img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

# build cubical complex
cc_tuple = image_to_cubical_complex(img)
cc = cc_tuple[0] # Extract the CubicalComplex object

# compute persistence
H0, H1 = compute_persistence(cc)

# combine H0 and H1
diagram = [(0, tuple(bd)) for bd in H0] + [(1, tuple(bd)) for bd in H1]# inorder to plot with two different colors, if we use concatanate then it would be same color for both h0 and h1

# plot
plt.figure()
gd.plot_persistence_diagram(diagram)
plt.title("Persistence Diagram with Tumor")
plt.show()

In [ ]:
path = no_paths[0]

# load image
img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

# build cubical complex
cc_tuple = image_to_cubical_complex(img)
cc = cc_tuple[0] # Extract the CubicalComplex object

# compute persistence
H0, H1 = compute_persistence(cc)

# combine H0 and H1
diagram = [(0, tuple(bd)) for bd in H0] + [(1, tuple(bd)) for bd in H1]

# plot
plt.figure()
gd.plot_persistence_diagram(diagram)
plt.title("Persistence Diagram without tumor")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=300000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_log))

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

rbf is radial basis function which allows model to run by non linear patterns, C is regularisation parameter, small c is for more generalised, less overfitting model and large c fits the data closely(overfitting prob)

In [ ]:
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

n estimators is no.of decision trees

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
log_acc = accuracy_score(y_test, y_pred_log)
svm_acc = accuracy_score(y_test, y_pred_svm)
rf_acc  = accuracy_score(y_test, y_pred_rf)

In [ ]:
import matplotlib.pyplot as plt

models = ['Logistic Regression', 'SVM', 'Random Forest']
accuracies = [log_acc, svm_acc, rf_acc]

plt.figure(figsize=(8,5))
plt.bar(models, accuracies)

plt.title("Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")

# show values on top
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha='center')

plt.ylim(0, 1)
plt.show()

In [ ]:
# combining paths and labels
all_paths = yes_paths + no_paths
all_labels = [1]*len(yes_paths) + [0]*len(no_paths)

# pick 50 random samples
indices_50 = random.sample(range(len(all_paths)), 50)

# pick 5 from those 50
indices_5 = random.sample(indices_50, 5)

print("Random Testing on 5 Images\n")

for idx in indices_5:
    path = all_paths[idx]
    true_label = all_labels[idx]

    # show image
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    plt.imshow(img, cmap='gray')
    plt.title(f"Actual: {'Tumor' if true_label==1 else 'No Tumor'}")
    plt.axis('off')
    plt.show()

    # extract features
    features = process_image(path)
    features = features.reshape(1, -1)

    # predict
    pred = rf_model.predict(features)[0]

    print("Prediction:", "Tumor" if pred==1 else "No Tumor")

In [ ]:
from google.colab import files

# upload file
uploaded = files.upload()

# get uploaded file name
file_name = list(uploaded.keys())[0]

# read image
img = cv2.imread(file_name, cv2.IMREAD_GRAYSCALE)

# display image
plt.imshow(img, cmap='gray')
plt.title("Uploaded MRI Scan")
plt.axis('off')
plt.show()

# extract features
features = process_image(file_name)
features = np.array(features).reshape(1, -1)

# predict
prediction = rf_model.predict(features)[0]
prob = rf_model.predict_proba(features)[0]

# output result
if prediction == 1:
    print("Prediction: Brain Tumor Detected")
else:
    print("Prediction: No Brain Tumor Detected")